# Min-K% Prob — Master Notebook

This notebook orchestrates the full Min-K% Prob replication pipeline.
Each module calls a script in `scripts/` which in turn calls `src/`.
No logic is written inline here — the notebook is a runner, not a codebase.

**Isolation contract:**
- `src/` — pure Python logic (imported by scripts, tested by pytest)
- `scripts/` — orchestration, CLI entry points, Drive I/O
- This notebook — runs scripts, displays outputs, writes nothing to `src/`

---

## [0] Runtime and GPU Check

Verifies the Colab runtime has GPU access and confirms the Python environment before any module runs.

In [1]:
import torch, sys
print(f'Python  : {sys.version.split()[0]}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
else:
    print('GPU     : None — switch Runtime > Change runtime type > GPU')

Python  : 3.12.13
PyTorch : 2.11.0+cu128
CUDA    : True
GPU     : Tesla T4


## [1] Drive Mount and Repository Setup

Mounts the shared Google Drive folder and clones the repository onto the Colab instance.
The `DRIVE` variable is set here and exported as an environment variable so all downstream scripts resolve their output paths without hardcoding.

> **First-time setup:** create the shared folder in Drive and share it with all team members before running this cell.

In [2]:
import os
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Shared Drive folder (all members use the same path once shared)
DRIVE = '/content/drive/MyDrive/min-k-project-group2'
os.environ['DRIVE'] = DRIVE
Path(DRIVE).mkdir(parents=True, exist_ok=True)
print(f'Drive mounted at : {DRIVE}')

# Set environment variable for downstream scripts
%env DRIVE=/content/drive/MyDrive/min-k-project-group2

# Repo clone / pull
REPO_URL = "https://github.com/dazzlear/ai-final-project.git"
BRANCH = "hanna/module-3"
PROJECT_DIR = "/content/ai-final-project"

if not Path(PROJECT_DIR).exists():
    !git clone {REPO_URL} {PROJECT_DIR}

%cd {PROJECT_DIR}
!git checkout {BRANCH}
!git pull origin {BRANCH}
print(f'Repo ready at : {PROJECT_DIR}')

Mounted at /content/drive
Drive mounted at : /content/drive/MyDrive/min-k-project-group2
env: DRIVE=/content/drive/MyDrive/min-k-project-group2
Cloning into '/content/ai-final-project'...
remote: Enumerating objects: 466, done.
remote: Counting objects: 100% (147/147), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 466 (delta 70), reused 65 (delta 28), pack-reused 319 (from 1)
Receiving objects: 100% (466/466), 637.73 KiB | 3.22 MiB/s, done.
Resolving deltas: 100% (234/234), done.
/content/ai-final-project
Branch 'hanna/module-3' set up to track remote branch 'hanna/module-3' from 'origin'.
Switched to a new branch 'hanna/module-3'
From https://github.com/dazzlear/ai-final-project
 * branch            hanna/module-3 -> FETCH_HEAD
Already up to date.
Repo ready at : /content/ai-final-project


## [2] Configuration

**This is the only cell that should be edited between runs.**
All scripts read these variables via CLI arguments or the `DRIVE` environment variable.

| Variable | Purpose |
|---|---|
| `LENGTHS` | WikiMIA length buckets to process |
| `DEFAULT_MODEL` | Model used for smoke test and full runs |
| `SAMPLE_SIZE` | Rows per sample CSV (balanced: half per label) |
| `DRIVE` | Root of the shared Drive output folder |

In [3]:
# Edit this cell only

LENGTHS       = [32, 64, 128, 256]   # subset e.g. [64] for a quick test
DEFAULT_MODEL = 'EleutherAI/pythia-410m'
SAMPLE_SIZE   = 10

MODELS = {
    'pythia410m' : 'EleutherAI/pythia-410m',    # smoke-test model only
    'pythia2.8b' : 'EleutherAI/pythia-2.8b',
    'gptneo1.3b' : 'EleutherAI/gpt-neo-1.3B',
    'opt1.3b'    : 'facebook/opt-1.3b',
}
SMOKE_MODEL = 'pythia410m'

# Derived paths -- do not edit
DRIVE_01 = f'{DRIVE}/01_dataset'
DRIVE_02 = f'{DRIVE}/02_model_loading'
DRIVE_03 = f'{DRIVE}/03_mink_scores'
DRIVE_04 = f'{DRIVE}/04_baseline_scores'
DRIVE_05 = f'{DRIVE}/05_evaluation'

from pathlib import Path
for _d in [DRIVE_01, DRIVE_02, DRIVE_03, DRIVE_04, DRIVE_05]:
    Path(_d).mkdir(parents=True, exist_ok=True)

print('Config set:')
print(f'  LENGTHS       = {LENGTHS}')
print(f'  DEFAULT_MODEL = {DEFAULT_MODEL}')
print(f'  DRIVE_01      = {DRIVE_01}')

Config set:
  LENGTHS       = [32, 64, 128, 256]
  DEFAULT_MODEL = EleutherAI/pythia-410m
  DRIVE_01      = /content/drive/MyDrive/min-k-project-group2/01_dataset


## [3] Install Dependencies

Installs all required packages. This cell is idempotent — safe to re-run.

In [4]:
!pip install -q datasets transformers torch scikit-learn pandas numpy matplotlib tqdm sentencepiece protobuf tiktoken

## [4] Module 01 — Original Dataset Loading and Preparation

This module creates only the original WikiMIA processed and sample CSVs. It does **not** create paraphrases and does not add `paraphrase_text` to the processed files.

**Original files (kept separate and unchanged by Module 1B):**

- `wikimia_length{N}_processed.csv`
- `wikimia_length{N}_sample.csv`
- `summaries/dataset_summary_len{N}.txt`
- `summaries/dataset_summary_all.csv`


In [ ]:
# Create only missing processed splits. Existing processed CSVs are not overwritten.
from pathlib import Path

missing_lengths = [
    length for length in LENGTHS
    if not (Path(DRIVE_01) / f'wikimia_length{length}_processed.csv').exists()
]

if missing_lengths:
    _missing_lengths_arg = ' '.join(str(length) for length in missing_lengths)
    print('Creating missing processed lengths:', missing_lengths)
    !python scripts/run_01_dataset.py \
        --lengths {_missing_lengths_arg} \
        --output_dir {DRIVE_01} \
        --sample_size {SAMPLE_SIZE}
else:
    print('All processed CSVs already exist. Module 01 generation skipped.')


Creating missing processed lengths: [32, 64, 128, 256]

Output directory : /content/drive/MyDrive/daz-test-min-k-project-group2/01_dataset
Lengths          : [32, 64, 128, 256]
Sample size      : 10

  Processing WikiMIA_length32
data/WikiMIA_length128-00000-of-00001-ff(…): 100% 132k/132k [00:01<00:00, 84.3kB/s]
data/WikiMIA_length256-00000-of-00001-e9(…): 100% 92.9k/92.9k [00:00<00:00, 227kB/s]
data/WikiMIA_length32-00000-of-00001-6d3(…): 100% 100k/100k [00:00<00:00, 262kB/s]
data/WikiMIA_length64-00000-of-00001-c33(…): 100% 140k/140k [00:00<00:00, 360kB/s]
Generating WikiMIA_length128 split: 100% 250/250 [00:00<00:00, 8591.15 examples/s]
Generating WikiMIA_length256 split: 100% 82/82 [00:00<00:00, 28955.46 examples/s]
Generating WikiMIA_length32 split: 100% 776/776 [00:00<00:00, 308042.77 examples/s]
Generating WikiMIA_length64 split: 100% 542/542 [00:00<00:00, 182668.76 examples/s]
  Saved processed CSV  : /content/drive/MyDrive/daz-test-min-k-project-group2/01_dataset/wikimia_lengt

In [ ]:
from pathlib import Path

script_path = Path(
    "/content/ai-final-project/scripts/run_01b_paraphrase.py"
)

print("Script exists:", script_path.exists())
print("Script path:", script_path)

Script exists: True
Script path: /content/ai-final-project/scripts/run_01b_paraphrase.py


### 4.1 Module 1B — Separate Paraphrase CSV Generation

Creates these files without modifying the processed CSVs:

- `wikimia_length32_paraphrased.csv`
- `wikimia_length64_paraphrased.csv`
- `wikimia_length128_paraphrased.csv`
- `wikimia_length256_paraphrased.csv`

Every paraphrase CSV is standardized to exactly:

`text_id, original_text, label, paraphrase_text`

The existing length-64 paraphrase file is reused. If it uses the old schema (`text` as the paraphrase and `original_text` as the source), the script converts it safely and creates a backup under `backups_paraphrase/`.


In [ ]:
from pathlib import Path
import hashlib
import pandas as pd

PARAPHRASE_SCRIPT = Path(PROJECT_DIR) / 'scripts' / 'run_01b_paraphrase.py'
if not PARAPHRASE_SCRIPT.exists():
    raise FileNotFoundError(
        f'{PARAPHRASE_SCRIPT} is missing. Add and commit run_01b_paraphrase.py first.'
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

processed_paths = {
    length: Path(DRIVE_01) / f'wikimia_length{length}_processed.csv'
    for length in LENGTHS
}
missing = [str(path) for path in processed_paths.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Missing processed CSVs: ' + ', '.join(missing))

processed_hashes_before = {
    length: sha256_file(path) for length, path in processed_paths.items()
}
print('Processed CSV hashes recorded. These files must remain unchanged.')
for length, path in processed_paths.items():
    print(f'  len{length}: {len(pd.read_csv(path))} rows | {processed_hashes_before[length][:12]}...')


Processed CSV hashes recorded. These files must remain unchanged.
  len32: 776 rows | ed1e07403ea9...
  len64: 542 rows | 56a2ada25d76...
  len128: 250 rows | d6499d9d1118...
  len256: 82 rows | 7a0d89e1db40...


#### Generate or resume all paraphrase files

This command is resumable. If Colab disconnects, rerun it; completed `paraphrase_text` values are reused.


In [ ]:
_lengths_arg = ' '.join(str(length) for length in LENGTHS)

!python scripts/run_01b_paraphrase.py \
    --input_dir {DRIVE_01} \
    --output_dir {DRIVE_01} \
    --lengths {_lengths_arg} \
    --model_name Vamsi/T5_Paraphrase_Paws \
    --batch_size 4 \
    --save_every 1 \
    --require_complete



WikiMIA length 32
Processed (read only): /content/drive/MyDrive/daz-test-min-k-project-group2/01_dataset/wikimia_length32_processed.csv
Paraphrase output    : /content/drive/MyDrive/daz-test-min-k-project-group2/01_dataset/wikimia_length32_paraphrased.csv
Rows in processed split : 776
Existing paraphrases    : 0
Rows selected to create : 776
config.json: 100% 1.21k/1.21k [00:00<00:00, 4.72MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 160kB/s]
spiece.model: 100% 792k/792k [00:00<00:00, 111MB/s]
special_tokens_map.json: 100% 1.79k/1.79k [00:00<00:00, 4.85MB/s]
model.safetensors: 100% 892M/892M [00:13<00:00, 65.9MB/s]
Loading weights: 100% 257/257 [00:00<00:00, 13286.69it/s]
Paraphrase model: Vamsi/T5_Paraphrase_Paws
Device: cuda
length 32: 100% 776/776 [04:50<00:00,  2.67it/s]
Saved: /content/drive/MyDrive/daz-test-min-k-project-group2/01_dataset/wikimia_length32_paraphrased.csv
Remaining empty paraphrases: 0

WikiMIA length 64
Processed (read only): /content/drive/MyDrive/d

#### Final validation and processed-file integrity check

Checks nulls, empty values, duplicate IDs, row-count parity, matching ID sets, label alignment, original-text alignment, and one-to-one matching by `text_id`. It also confirms that the original processed CSVs were not changed.


In [ ]:
!python scripts/run_01b_paraphrase.py \
    --input_dir {DRIVE_01} \
    --output_dir {DRIVE_01} \
    --lengths {_lengths_arg} \
    --validate_only \
    --require_complete

processed_hashes_after = {
    length: sha256_file(path) for length, path in processed_paths.items()
}
changed_processed = [
    length for length in LENGTHS
    if processed_hashes_before[length] != processed_hashes_after[length]
]
if changed_processed:
    raise AssertionError(
        f'Processed CSVs changed unexpectedly for lengths: {changed_processed}'
    )
print('✅ Processed CSVs remained unchanged.')

summary_path = Path(DRIVE_01) / 'summaries' / 'paraphrase_validation_all.csv'
validation_summary = pd.read_csv(summary_path)
display(validation_summary)



Running validation...
Length 32: PASS | rows=776/776 | empty paraphrases=0
Length 64: PASS | rows=542/542 | empty paraphrases=0
Length 128: PASS | rows=250/250 | empty paraphrases=0
Length 256: PASS | rows=82/82 | empty paraphrases=0
Validation summary: /content/drive/MyDrive/daz-test-min-k-project-group2/01_dataset/summaries/paraphrase_validation_all.csv

PASS: all requested paraphrase CSVs satisfy the validation rules.
✅ Processed CSVs remained unchanged.


,length,processed_rows,paraphrase_rows,exact_standard_columns,missing_required_columns,null_text_id,empty_text_id,duplicate_text_id,null_original_text,empty_original_text,...,empty_paraphrase_text,row_count_parity,text_id_set_match,text_id_order_match,one_to_one_text_id,label_alignment,original_text_alignment,identical_paraphrase_count,status,reason
0,32,776,776,True,NaN,0,0,0,0,0,...,0,True,True,True,True,True,True,2,PASS,NaN
1,64,542,542,True,NaN,0,0,0,0,0,...,0,True,True,True,True,True,True,0,PASS,NaN
2,128,250,250,True,NaN,0,0,0,0,0,...,0,True,True,True,True,True,True,0,PASS,NaN
3,256,82,82,True,NaN,0,0,0,0,0,...,0,True,True,True,True,True,True,0,PASS,NaN


#### Preview standardized paraphrase outputs


In [ ]:
EXPECTED_COLUMNS = ['text_id', 'original_text', 'label', 'paraphrase_text']

for length in LENGTHS:
    paraphrase_path = Path(DRIVE_01) / f'wikimia_length{length}_paraphrased.csv'
    processed_path = Path(DRIVE_01) / f'wikimia_length{length}_processed.csv'
    df_para = pd.read_csv(paraphrase_path)
    df_original = pd.read_csv(processed_path)
    assert df_para.columns.tolist() == EXPECTED_COLUMNS
    assert 'paraphrase_text' not in df_original.columns
    print(f'\nLength {length}: {len(df_para)} paraphrase rows | columns={df_para.columns.tolist()}')
    display(df_para.head(3))



Length 32: 776 paraphrase rows | columns=['text_id', 'original_text', 'label', 'paraphrase_text']


,text_id,original_text,label,paraphrase_text
0,0,The 12th Circle Chart Music Awards ceremony wa...,0,The 12th Circle Chart Music Awards ceremony wa...
1,1,Hurricane Ana was the second tropical cyclone ...,1,Hurricane Ana was the second tropical cyclone ...
2,2,On 6 February 2023 a referendum was held in th...,0,On 6 February 2023 a referendum was held in th...



Length 64: 542 paraphrase rows | columns=['text_id', 'original_text', 'label', 'paraphrase_text']


,text_id,original_text,label,paraphrase_text
0,0,The 12th Circle Chart Music Awards ceremony wa...,0,The 12th Circle Chart Music Awards ceremony wa...
1,1,Hurricane Ana was the second tropical cyclone ...,1,Hurricane Ana was the second tropical cyclone ...
2,2,On 6 February 2023 a referendum was held in th...,0,On 6 February 2023 a referendum was held in th...



Length 128: 250 paraphrase rows | columns=['text_id', 'original_text', 'label', 'paraphrase_text']


,text_id,original_text,label,paraphrase_text
0,0,Hurricane Ana was the second tropical cyclone ...,1,Hurricane Ana was the second tropical cyclone ...
1,1,On 6 February 2023 a referendum was held in th...,0,On 6 February 2023 a referendum was held in th...
2,2,"The 2013–2016 epidemic of Ebola virus disease,...",1,The 2013–2016 epidemic of Ebola virus disease ...



Length 256: 82 paraphrase rows | columns=['text_id', 'original_text', 'label', 'paraphrase_text']


,text_id,original_text,label,paraphrase_text
0,0,Hurricane Ana was the second tropical cyclone ...,1,Hurricane Ana was the second tropical cyclone ...
1,1,"The 2013–2016 epidemic of Ebola virus disease,...",1,The 2013–2016 epidemic of Ebola virus disease ...
2,2,The case of Ashya King concerns a boy named As...,1,The case of Ashya King concerns a boy named As...


### 4.2 Original Dataset Output Preview

Displays the original processed splits. These files remain separate from the paraphrase outputs.


In [ ]:
from pathlib import Path
import pandas as pd

for length in LENGTHS:
    path = Path(DRIVE_01) / f'wikimia_length{length}_processed.csv'
    if path.exists():
        df = pd.read_csv(path)
        print(f'\n{"="*55}')
        print(f'  WikiMIA_length{length}  --  {len(df)} rows')
        print(f'{"="*55}')
        display(df.head(3))
        vc = df['label'].value_counts().rename({0: 'non-member (0)', 1: 'member (1)'})
        print(vc.to_string())
    else:
        print(f'[MISSING] {path}')


  WikiMIA_length32  --  776 rows


,text_id,text,label
0,0,The 12th Circle Chart Music Awards ceremony wa...,0
1,1,Hurricane Ana was the second tropical cyclone ...,1
2,2,On 6 February 2023 a referendum was held in th...,0


label
non-member (0)    389
member (1)        387

  WikiMIA_length64  --  542 rows


,text_id,text,label
0,0,The 12th Circle Chart Music Awards ceremony wa...,0
1,1,Hurricane Ana was the second tropical cyclone ...,1
2,2,On 6 February 2023 a referendum was held in th...,0


label
member (1)        284
non-member (0)    258

  WikiMIA_length128  --  250 rows


,text_id,text,label
0,0,Hurricane Ana was the second tropical cyclone ...,1
1,1,On 6 February 2023 a referendum was held in th...,0
2,2,"The 2013–2016 epidemic of Ebola virus disease,...",1


label
member (1)        139
non-member (0)    111

  WikiMIA_length256  --  82 rows


,text_id,text,label
0,0,Hurricane Ana was the second tropical cyclone ...,1
1,1,"The 2013–2016 epidemic of Ebola virus disease,...",1
2,2,The case of Ashya King concerns a boy named As...,1


label
member (1)        51
non-member (0)    31


### 4.3 Module 01 Sanity Check

Validates the original processed datasets independently of the paraphrase files.


In [ ]:
from pathlib import Path
import pandas as pd

PASS = True

for length in LENGTHS:
    processed = Path(DRIVE_01) / f'wikimia_length{length}_processed.csv'
    sample    = Path(DRIVE_01) / f'wikimia_length{length}_sample.csv'
    summary   = Path(DRIVE_01) / 'summaries' / f'dataset_summary_len{length}.txt'

    checks = {
        f'processed CSV exists (len{length})': processed.exists(),
        f'sample CSV exists    (len{length})': sample.exists(),
        f'summary txt exists   (len{length})': summary.exists(),
    }

    if processed.exists():
        df = pd.read_csv(processed)
        checks[f'no missing text   (len{length})'] = df['text'].isna().sum() == 0
        checks[f'no missing labels (len{length})'] = df['label'].isna().sum() == 0
        checks[f'no empty texts    (len{length})'] = (df['text'].str.strip() == '').sum() == 0

        label_0 = int((df['label'] == 0).sum())
        label_1 = int((df['label'] == 1).sum())
        larger  = max(label_0, label_1)
        smaller = min(label_0, label_1)
        ratio   = smaller / larger

        if ratio < 0.90:
            # Imbalance is expected for longer lengths due to upstream filtering.
            # Logged as a warning — does not fail the pipeline.
            print(f'  ⚠️   label imbalance (len{length}): {label_0} vs {label_1} '
                  f'(ratio {ratio:.2f}) — known upstream dataset property')
        else:
            checks[f'labels balanced (len{length}) [{label_0} vs {label_1}]'] = True

    for name, ok in checks.items():
        icon = '✅' if ok else '❌'
        print(f'  {icon}  {name}')
        if not ok:
            PASS = False

master = Path(DRIVE_01) / 'summaries' / 'dataset_summary_all.csv'
icon = '✅' if master.exists() else '❌'
print(f'  {icon}  master summary CSV exists')
if not master.exists():
    PASS = False

print()
if PASS:
    print('✅  Module 01 PASSED — proceed to Module 02.')
else:
    print('❌  Module 01 FAILED — fix the issues above before running Module 02.')

  ✅  processed CSV exists (len32)
  ✅  sample CSV exists    (len32)
  ✅  summary txt exists   (len32)
  ✅  no missing text   (len32)
  ✅  no missing labels (len32)
  ✅  no empty texts    (len32)
  ✅  labels balanced (len32) [389 vs 387]
  ✅  processed CSV exists (len64)
  ✅  sample CSV exists    (len64)
  ✅  summary txt exists   (len64)
  ✅  no missing text   (len64)
  ✅  no missing labels (len64)
  ✅  no empty texts    (len64)
  ✅  labels balanced (len64) [258 vs 284]
  ⚠️   label imbalance (len128): 111 vs 139 (ratio 0.80) — known upstream dataset property
  ✅  processed CSV exists (len128)
  ✅  sample CSV exists    (len128)
  ✅  summary txt exists   (len128)
  ✅  no missing text   (len128)
  ✅  no missing labels (len128)
  ✅  no empty texts    (len128)
  ⚠️   label imbalance (len256): 31 vs 51 (ratio 0.61) — known upstream dataset property
  ✅  processed CSV exists (len256)
  ✅  sample CSV exists    (len256)
  ✅  summary txt exists   (len256)
  ✅  no missing text   (len256)
  ✅  no 

### 4.4 Module 01 Completion Checklist

Module 01 is complete only when all of the following are true:

- The four original `wikimia_length{N}_processed.csv` files still exist and remain unchanged.
- The four separate `wikimia_length{N}_paraphrased.csv` files exist.
- Every paraphrase CSV has exactly these columns: `text_id`, `original_text`, `label`, `paraphrase_text`.
- The final validation report shows `PASS` for lengths 32, 64, 128, and 256.
- There are no null or empty paraphrases.
- Every `text_id` appears exactly once and matches one row in the corresponding processed CSV.
- Labels and original text match the corresponding processed split.

Generated dataset files stay in Google Drive. Commit only the script and notebook to GitHub unless the group explicitly asks you to commit the generated CSV files.


In [ ]:
from pathlib import Path
import pandas as pd

summary_path = Path(DRIVE_01) / 'summaries' / 'paraphrase_validation_all.csv'
if not summary_path.exists():
    raise FileNotFoundError(f'Validation report not found: {summary_path}')

summary = pd.read_csv(summary_path)
required_lengths = {32, 64, 128, 256}
actual_lengths = set(summary['length'].astype(int))

if actual_lengths != required_lengths:
    raise AssertionError(
        f'Validation report lengths do not match. Expected {required_lengths}, got {actual_lengths}'
    )

failed = summary.loc[summary['status'] != 'PASS']
if not failed.empty:
    display(failed)
    raise AssertionError('One or more paraphrase CSVs failed validation.')

print('✅ DAZEL MODULE 01 TASK COMPLETE')
print('✅ Four separate paraphrase CSVs were created and validated.')
print('✅ Original processed CSVs were not modified.')
display(summary)


✅ DAZEL MODULE 01 TASK COMPLETE
✅ Four separate paraphrase CSVs were created and validated.
✅ Original processed CSVs were not modified.


,length,processed_rows,paraphrase_rows,exact_standard_columns,missing_required_columns,null_text_id,empty_text_id,duplicate_text_id,null_original_text,empty_original_text,...,empty_paraphrase_text,row_count_parity,text_id_set_match,text_id_order_match,one_to_one_text_id,label_alignment,original_text_alignment,identical_paraphrase_count,status,reason
0,32,776,776,True,NaN,0,0,0,0,0,...,0,True,True,True,True,True,True,2,PASS,NaN
1,64,542,542,True,NaN,0,0,0,0,0,...,0,True,True,True,True,True,True,0,PASS,NaN
2,128,250,250,True,NaN,0,0,0,0,0,...,0,True,True,True,True,True,True,0,PASS,NaN
3,256,82,82,True,NaN,0,0,0,0,0,...,0,True,True,True,True,True,True,0,PASS,NaN


## [5] Module 02 — Model Loading and Verification

Calls `scripts/run_02_model_loading.py` to load each model in the `MODELS`
config, run a diagnostics check, and save a verification report to Drive.
No log-probabilities are computed here — that is Module 03.

**What this section produces** (written to `{DRIVE}/02_model_loading/`):

| File | Description |
|---|---|
| `model_verification_report.csv` | One row per model: architecture, param count, VRAM, vocab size, roundtrip and forward-pass results |

The script loads each model, captures diagnostics, runs a tokenization
roundtrip and a single forward pass to confirm the model is functional,
then unloads it before moving to the next. All four models are verified
in sequence.

> **Before running:** confirm Module 01 passed (all ✅ in §4.2).

In [ ]:
# ── Module 02 run ─────────────────────────────────────────────────────────────
# Verifies all models in MODELS can load and produce logits.
# Outputs: model_verification_report.csv in DRIVE_02.

_models_arg = ' '.join(MODELS.values())

!python scripts/run_02_model_loading.py \
    --models {_models_arg} \
    --output_dir {DRIVE_02}


════════════════════════════════════════════════════════════
  Module 02 — Model Loading and Verification
  Models to verify: 4
════════════════════════════════════════════════════════════
[load_model] Loading 'EleutherAI/pythia-410m' ...
config.json: 100% 570/570 [00:00<00:00, 2.63MB/s]
model.safetensors: 100% 911M/911M [00:09<00:00, 93.5MB/s]
Loading weights: 100% 292/292 [00:00<00:00, 1194.83it/s]
tokenizer_config.json: 100% 396/396 [00:00<00:00, 1.44MB/s]
tokenizer.json: 100% 2.11M/2.11M [00:00<00:00, 131MB/s]
special_tokens_map.json: 100% 99.0/99.0 [00:00<00:00, 633kB/s]
[load_model] Done.

  Model        : EleutherAI/pythia-410m
  Architecture : GPTNeoXForCausalLM
  Parameters   : 405.3M
  Device       : cuda:0
  VRAM used    : 811 MB
  Vocab size   : 50,254
  Roundtrip    : ✅
  Forward pass : ✅  (logits shape: [1, 10, 50304])
  GPU cache cleared.
[load_model] Loading 'EleutherAI/pythia-2.8b' ...
config.json: 100% 571/571 [00:00<00:00, 3.02MB/s]
model.safetensors: 100% 5.68G/5.6

### 5.1 Output Preview

Displays the verification report table. Every model should show
`roundtrip_ok = True` and `forward_pass_ok = True` before proceeding.
VRAM readings confirm the model fit in GPU memory without OOM.

In [ ]:
from pathlib import Path
import pandas as pd

report_path = Path(DRIVE_02) / 'model_verification_report.csv'

if report_path.exists():
    df_report = pd.read_csv(report_path)
    display(df_report[[
        'model_key', 'architecture', 'parameters_M',
        'vram_mb', 'vocab_size', 'roundtrip_ok', 'forward_pass_ok'
    ]])

    failed = df_report[
        ~df_report['roundtrip_ok'] | ~df_report['forward_pass_ok']
    ]
    if len(failed) == 0:
        print('\nAll models passed verification ✅')
    else:
        print('\n⚠️  Failed models:')
        print(failed[['model_key', 'roundtrip_ok', 'forward_pass_ok']].to_string(index=False))
else:
    print(f'[MISSING] {report_path}')
    print('Re-run the cell above.')

,model_key,architecture,parameters_M,vram_mb,vocab_size,roundtrip_ok,forward_pass_ok
0,pythia410m,GPTNeoXForCausalLM,405.334016,810.669056,50254,True,True
1,pythia28b,GPTNeoXForCausalLM,2775.208960,5559.987200,50254,True,True
2,gptneo13B,GPTNeoForCausalLM,1315.575808,5372.534784,50257,True,True
3,opt13b,OPTForCausalLM,1315.758080,2641.084416,50265,True,True



All models passed verification ✅


### 5.2 Module 02 Sanity Check

Verifies the verification report exists and that every configured model
passed both checks before proceeding to Module 03.

**Pass criteria (all must be green):**
- `model_verification_report.csv` present
- One row per model in `MODELS`
- All rows: `roundtrip_ok = True`
- All rows: `forward_pass_ok = True`

In [ ]:
from pathlib import Path
import pandas as pd

PASS = True
report_path = Path(DRIVE_02) / 'model_verification_report.csv'

# ── Report file exists ────────────────────────────────────────────────────────
report_ok = report_path.exists()
print(f'  {"✅" if report_ok else "❌"}  model_verification_report.csv exists')
if not report_ok:
    PASS = False
else:
    df_r = pd.read_csv(report_path)

    # ── Row count matches MODELS ──────────────────────────────────────────────
    expected_n = len(MODELS)
    found_n    = len(df_r)
    count_ok   = found_n == expected_n
    print(f'  {"✅" if count_ok else "❌"}  row count: {found_n} / {expected_n} expected')
    if not count_ok:
        PASS = False

    # ── Per-model checks ──────────────────────────────────────────────────────
    print()
    for _, row in df_r.iterrows():
        rt_ok = bool(row['roundtrip_ok'])
        fp_ok = bool(row['forward_pass_ok'])
        icon  = '✅' if (rt_ok and fp_ok) else '❌'
        print(f'  {icon}  {row["model_key"]:15s}  '
              f'roundtrip={rt_ok}  forward_pass={fp_ok}  '
              f'{row["parameters_M"]:.0f}M params  {row["vram_mb"]:.0f} MB VRAM')
        if not (rt_ok and fp_ok):
            PASS = False

print()
if PASS:
    print('✅  Module 02 PASSED — proceed to Module 03.')
else:
    print('❌  Module 02 FAILED — fix the issues above before running Module 03.')

  ✅  model_verification_report.csv exists
  ✅  row count: 4 / 4 expected

  ✅  pythia410m       roundtrip=True  forward_pass=True  405M params  811 MB VRAM
  ✅  pythia28b        roundtrip=True  forward_pass=True  2775M params  5560 MB VRAM
  ✅  gptneo13B        roundtrip=True  forward_pass=True  1316M params  5373 MB VRAM
  ✅  opt13b           roundtrip=True  forward_pass=True  1316M params  2641 MB VRAM

✅  Module 02 PASSED — proceed to Module 03.


## [6] Module 03 — Min-K% Prob Scoring

Calls `scripts/run_03_mink.py` to compute Min-K% Prob scores for every
configured **model × length × setting** combination.

**What this module does:**
The Min-K% Prob method (Shi et al., ICLR 2024, Eq. 1) detects whether a text
was seen during LLM pretraining by examining the *lowest-probability tokens*
in that text.  The intuition: a model assigns higher probabilities to tokens
it has memorised; unseen text is more likely to contain surprising (low-prob)
outlier tokens.

**Four pipeline stages:**

| Stage | Function | Description |
|-------|----------|-------------|
| 1 | `tokenize_text()` | Split text into subword tokens |
| 2 | `compute_token_logprobs()` | Forward pass → per-token log p |
| 3 | `select_min_k_tokens()` | Pick bottom k% by log probability |
| 4 | `min_k_prob()` | Average log-prob of selected set |

**Output CSV schema** (text_id, text, label, min_k_score, n_tokens, n_selected) — locked for Modules 04 and 05.

---

### 6.0 Stage-by-Stage Visualization

Load pythia-410m (smallest model) and walk through all four pipeline stages on sample texts for understanding.
This cell does NOT affect the full-run outputs.

In [5]:
import sys, os
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Ensure src/ is on path
_src = os.path.join(os.environ.get('PROJECT_DIR', '/content/ai-final-project'), 'src')
if _src not in sys.path:
    sys.path.insert(0, _src)

from models  import load_model
from methods import select_min_k_tokens, min_k_prob

# Which log-probability backend to use for the stage-by-stage demo below.
# "manual" — plain-Python softmax/log/gather, shown to demonstrate the math
#            (used for the smoke test in 6.1).
# "auto"   — torch.nn.functional.log_softmax / Tensor.gather() (used for the
#            full run in 6.2).
VIZ_IMPLEMENTATION = 'manual'

if VIZ_IMPLEMENTATION == 'manual':
    from log_probability_compute_manual import tokenize_text, compute_token_logprobs
else:
    from log_probability_compute_auto import tokenize_text, compute_token_logprobs

print(f'Using "{VIZ_IMPLEMENTATION}" log-probability implementation for Stage 1-4 demo.')

VIZ_MODEL  = 'EleutherAI/pythia-410m'
VIZ_K      = 20
VIZ_LENGTH = 64
VIZ_N      = 3

Using "manual" log-probability implementation for Stage 1-4 demo.


### 6.0.1 Load Model

In [7]:
print(f'Loading {VIZ_MODEL} for stage-by-stage demo ...')
_viz_model, _viz_tokenizer = load_model(VIZ_MODEL)
print('Model loaded.')

Loading EleutherAI/pythia-410m for stage-by-stage demo ...
[load_model] Loading 'EleutherAI/pythia-410m' ...


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

[load_model] Done.
Model loaded.


In [9]:
"""### 6.0.x Select sample rows for visualization"""

# Use one of the processed datasets (pick a length consistent with VIZ_LENGTH)
demo_path = Path(DRIVE_01) / f'wikimia_length{VIZ_LENGTH}_processed.csv'
demo_df = pd.read_csv(demo_path)

# Sample VIZ_N rows, ideally with a mix of labels
viz_rows = demo_df.sample(n=VIZ_N, random_state=42).reset_index(drop=True)
display(viz_rows[['text_id', 'text', 'label']])

,text_id,text,label
0,360,"The 2014 FIBA World Championship for Women, th...",1
1,73,Breck David Lafave Bednar (17 March 1999 – 17 ...,1
2,353,Devu G. versus State Of Kerala & Ors. (2023) i...,0


### 6.0.2 Stage 1 — Tokenization

In [10]:
print('=' * 65)
print('STAGE 1 — Tokenization')
print('=' * 65)
print('Each text is split into subword tokens using the model\'s tokenizer.')
print('These token IDs are the inputs to the forward pass in Stage 2.\n')

_stage1_results = []
for _, row in viz_rows.iterrows():
    tok = tokenize_text(row['text'], _viz_tokenizer)
    _stage1_results.append(tok)

    label_str = 'MEMBER (1)' if row['label'] == 1 else 'NON-MEMBER (0)'
    print(f'text_id={row["text_id"]}  label={label_str}')
    print(f'  n_tokens     : {tok["n_tokens"]}')
    print(f'  was_truncated: {tok["was_truncated"]}')
    preview = list(zip(tok["input_ids"][:12], tok["tokens"][:12]))
    print(f'  token preview (first 12):')
    for tid, tstr in preview:
        print(f'    {tid:>6}  {repr(tstr)}')
    if tok["n_tokens"] > 12:
        print(f'    ... {tok["n_tokens"] - 12} more tokens ...')
    print()

STAGE 1 — Tokenization
Each text is split into subword tokens using the model's tokenizer.
These token IDs are the inputs to the forward pass in Stage 2.

text_id=360  label=MEMBER (1)
  n_tokens     : 89
  was_truncated: False
  token preview (first 12):
       510  'The'
      4059  ' 2014'
       401  ' F'
      5472  'IB'
        34  'A'
      3645  ' World'
     11218  ' Championship'
       323  ' for'
     10168  ' Women'
        13  ','
       253  ' the'
      1722  ' 17'
    ... 77 more tokens ...

text_id=73  label=MEMBER (1)
  n_tokens     : 92
  was_truncated: False
  token preview (first 12):
     16212  'Bre'
       777  'ck'
      5119  ' David'
     35735  ' Laf'
      1123  'ave'
     19761  ' Bed'
     27380  'nar'
       313  ' ('
      1166  '17'
      3919  ' March'
      7544  ' 1999'
      1108  ' –'
    ... 80 more tokens ...

text_id=353  label=NON-MEMBER (0)
  n_tokens     : 78
  was_truncated: False
  token preview (first 12):
     11148  'Dev'
        86  '

### 6.0.3 Stage 2 — Token Log Probabilities

In [ ]:
print('=' * 65)
print('STAGE 2 — Token Log Probabilities')
print('=' * 65)
print('One forward pass returns log p(xi | x<i) for each predicted token.')
print('More negative = the token surprised the model.\n')

_stage2_results = []
for idx, (_, row) in enumerate(viz_rows.iterrows()):
    lp = compute_token_logprobs(row['text'], _viz_model, _viz_tokenizer)
    _stage2_results.append(lp)

    label_str = 'MEMBER (1)' if row['label'] == 1 else 'NON-MEMBER (0)'
    print(f'text_id={row["text_id"]}  label={label_str}')
    print(f'  n_predicted_tokens : {len(lp)}')
    print(f'  mean log-prob      : {np.mean(lp):.4f}')
    print(f'  min  log-prob      : {np.min(lp):.4f}')
    print(f'  max  log-prob      : {np.max(lp):.4f}')
    print(f'  first 8 values     : {[round(v,3) for v in lp[:8]]}')
    print()

# Visualise log-prob distributions
fig, axes = plt.subplots(1, VIZ_N, figsize=(5 * VIZ_N, 3), sharey=True)
colors = {1: '#2196F3', 0: '#FF5722'}

for ax, (_, row), lp in zip(axes, viz_rows.iterrows(), _stage2_results):
    c = colors[row['label']]
    ax.hist(lp, bins=20, color=c, alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.axvline(np.mean(lp), color='black', linestyle='--', linewidth=1.2,
               label=f'mean={np.mean(lp):.2f}')
    label_str = 'Member' if row['label'] == 1 else 'Non-Member'
    ax.set_title(f'text_id={row["text_id"]}  [{label_str}]', fontsize=10)
    ax.set_xlabel('log p(token | context)', fontsize=9)
    ax.legend(fontsize=8)

axes[0].set_ylabel('Token count', fontsize=9)
fig.suptitle('Stage 2 — Per-token log-probability distributions', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f'{DRIVE_03}/viz_stage2_logprob_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz_stage2_logprob_distributions.png')

### 6.0.4 Stage 3 — Select Bottom K% Tokens

In [ ]:
print('=' * 65)
print(f'STAGE 3 — Select Bottom {VIZ_K}% Tokens (Min-K% set)')
print('=' * 65)
print('Tokens are ranked by log-prob ascending (most surprising first).')
print(f'The bottom {VIZ_K}% by count are selected as the Min-K% set.\n')

_stage3_results = []
tok_preview_data = []

for idx, (_, row) in enumerate(viz_rows.iterrows()):
    lp   = _stage2_results[idx]
    tok  = _stage1_results[idx]

    sel_idx, sel_lp, rank_order = select_min_k_tokens(lp, k=VIZ_K)
    _stage3_results.append((sel_idx, sel_lp))

    label_str = 'MEMBER (1)' if row['label'] == 1 else 'NON-MEMBER (0)'
    print(f'text_id={row["text_id"]}  label={label_str}')
    print(f'  total tokens    : {len(lp)}')
    print(f'  k={VIZ_K}%  →  n_selected : {len(sel_idx)}')
    print(f'  selected token positions : {sel_idx}')

    sel_tokens = [tok['tokens'][i + 1] if i + 1 < len(tok['tokens']) else '<OOB>'
                  for i in sel_idx]
    print(f'  selected tokens  : {[repr(t) for t in sel_tokens[:10]]}{"..." if len(sel_tokens) > 10 else ""}')
    print(f'  selected log-probs: {[round(v,3) for v in sel_lp[:10]]}{"..." if len(sel_lp) > 10 else ""}')
    print(f'  mean of selected : {np.mean(sel_lp):.4f}')
    print()

    tok_preview_data.append({
        'text_id'   : row['text_id'],
        'label'     : row['label'],
        'lp'        : lp,
        'sel_idx'   : set(sel_idx),
    })

# Per-token bar chart with selected tokens highlighted
fig, axes = plt.subplots(VIZ_N, 1, figsize=(14, 3.5 * VIZ_N))
if VIZ_N == 1:
    axes = [axes]

for ax, data in zip(axes, tok_preview_data):
    lp      = data['lp']
    sel_set = data['sel_idx']
    n_show  = min(50, len(lp))

    bar_colors = ['#FF5722' if i in sel_set else '#B0BEC5' for i in range(n_show)]
    ax.bar(range(n_show), lp[:n_show], color=bar_colors, width=0.85, edgecolor='none')

    label_str = 'Member' if data['label'] == 1 else 'Non-Member'
    ax.set_title(f'text_id={data["text_id"]}  [{label_str}] — red = selected in Min-{VIZ_K}% set',
                 fontsize=10)
    ax.set_xlabel('Token position (first 50 shown)', fontsize=9)
    ax.set_ylabel('log p(token)', fontsize=9)
    ax.axhline(0, color='gray', linewidth=0.5)

    red_patch  = mpatches.Patch(color='#FF5722', label=f'Bottom {VIZ_K}% (selected)')
    grey_patch = mpatches.Patch(color='#B0BEC5', label='Remaining tokens')
    ax.legend(handles=[red_patch, grey_patch], fontsize=8, loc='lower right')

fig.suptitle(f'Stage 3 — Min-{VIZ_K}% token selection (red = selected)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{DRIVE_03}/viz_stage3_token_selection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: viz_stage3_token_selection.png')

### 6.0.5 Stage 4 — Min-K% Prob Score

In [ ]:
print('=' * 65)
print('STAGE 4 — Min-K% Prob Score (Equation 1)')
print('=' * 65)
print('The final score is the mean log-prob of selected (bottom-k%) tokens.')
print('Higher (less negative) = model assigns higher probs → more likely MEMBER.\n')

summary_rows = []
for idx, (_, row) in enumerate(viz_rows.iterrows()):
    lp    = _stage2_results[idx]
    score = min_k_prob(lp, k=VIZ_K)
    _, sel_lp = _stage3_results[idx]

    label_str = 'MEMBER (1)' if row['label'] == 1 else 'NON-MEMBER (0)'
    print(f'text_id={row["text_id"]}  label={label_str}')
    print(f'  all-token mean log-prob : {np.mean(lp):.4f}')
    print(f'  Min-K% Prob score       : {score:.4f}   <── detection score')
    print()
    summary_rows.append({
        'text_id'      : row['text_id'],
        'label'        : row['label'],
        'all_mean_lp'  : round(np.mean(lp), 4),
        'min_k_score'  : round(score, 4),
    })

print('Summary table:')
print(pd.DataFrame(summary_rows).to_string(index=False))
print()
print('Expected: member min_k_score > non-member min_k_score')
m1 = np.mean([r['min_k_score'] for r in summary_rows if r['label'] == 1])
m0 = np.mean([r['min_k_score'] for r in summary_rows if r['label'] == 0])
direction = '✅ correct direction' if m1 > m0 else '⚠️  unexpected — may be due to small sample'
print(f'  member mean    : {m1:.4f}')
print(f'  non-member mean: {m0:.4f}   {direction}')

del _viz_model, _viz_tokenizer
torch.cuda.empty_cache()
print('\nGPU cache cleared. Ready for full dataset run.')

### 6.1 Smoke Test

Validates the full script pipeline on 10 rows with the smallest model before committing to the multi-hour run.
Output: `mink_scores_pythia410m_len64_original.csv` (10 rows)

In [ ]:
_lengths_arg    = ' '.join(str(l) for l in LENGTHS)
_models_arg     = ' '.join(MODELS.values())
_model_keys_arg = ' '.join(MODELS.keys())
_settings_arg   = ' '.join(['original'])

_smoke_models_arg     = 'EleutherAI/pythia-410m'
_smoke_model_keys_arg = 'pythia410m'

!python scripts/run_03_mink_scores.py \
    --input_dir  {DRIVE_01} \
    --output_dir {DRIVE_03} \
    --models     {_smoke_models_arg} \
    --model_keys {_smoke_model_keys_arg} \
    --lengths    64 \
    --settings   original \
    --k          20 \
    --implementation manual \
    --smoke

print('\nSmoke test output preview:')
import pandas as pd
_smoke_csv = f'{DRIVE_03}/mink_scores_pythia410m_len64_original.csv'
df_smoke   = pd.read_csv(_smoke_csv)
display(df_smoke[['text_id', 'label', 'min_k_score', 'n_tokens', 'n_selected']])

### 6.2 Full Dataset Run

Computes scores for all models × lengths × settings combinations.
Estimated time on T4 GPU: 2–3 hours per large model.
Each CSV is written to Drive immediately; re-run with `--skip_existing` if interrupted.

In [ ]:
!python scripts/run_03_mink_scores.py \
    --input_dir     {DRIVE_01} \
    --output_dir    {DRIVE_03} \
    --models        {_models_arg} \
    --model_keys    {_model_keys_arg} \
    --lengths       {_lengths_arg} \
    --settings      {_settings_arg} \
    --k             20 \
    --implementation auto \
    --skip_existing \
    --cache_logprobs

### 6.3 Output Preview

Displays summary statistics (member mean vs non-member mean) for each model × length × setting.

In [ ]:
from pathlib import Path
import pandas as pd

_length = LENGTHS[0]

for model_key in MODELS:
    for setting in ['original', 'paraphrase']:
        csv_path = Path(DRIVE_03) / f'mink_scores_{model_key}_len{_length}_{setting}.csv'
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            m1 = df[df['label'] == 1]['min_k_score'].mean()
            m0 = df[df['label'] == 0]['min_k_score'].mean()
            print(f'\n{model_key}  |  len={_length}  |  {setting}  ({len(df)} rows)')
            print(f'  member mean    : {m1:.4f}')
            print(f'  non-member mean: {m0:.4f}')
            print()
            display(df[['text_id', 'label', 'min_k_score', 'n_tokens', 'n_selected']].head(6))
        else:
            print(f'[MISSING] {csv_path.name}')

### 6.4 Module 03 Sanity Check

Verifies all expected outputs exist and meet quality requirements.

**Pass criteria:**
- Score CSV exists for every model × length × setting
- Row count matches Module 01 processed CSV
- No NaN min_k_score values
- Score direction: member mean > non-member mean (at least on len=64 original)
- n_selected calculation is correct

In [ ]:
from pathlib import Path
import math
import pandas as pd

PASS = True
K_CHECK = 20

for model_key in MODELS:
    for length in LENGTHS:
        for setting in ['original', 'paraphrase']:
            csv_path = Path(DRIVE_03) / f'mink_scores_{model_key}_len{length}_{setting}.csv'
            ref_csv  = Path(DRIVE_01) / f'wikimia_length{length}_processed.csv'

            exists = csv_path.exists()
            icon   = '✅' if exists else '❌'
            label  = f'{model_key} | len={length} | {setting}'
            print(f'  {icon}  CSV exists         {label}')
            if not exists:
                PASS = False
                continue

            df = pd.read_csv(csv_path)

            if ref_csv.exists():
                expected = len(pd.read_csv(ref_csv))
                count_ok = len(df) == expected
                icon     = '✅' if count_ok else '⚠️ '
                print(f'  {icon}  row count          {label}: {len(df)} / {expected}')
                if not count_ok:
                    PASS = False

            nan_ok = df['min_k_score'].isna().sum() == 0
            icon   = '✅' if nan_ok else '❌'
            print(f'  {icon}  no NaN scores      {label}')
            if not nan_ok:
                PASS = False

            m1  = df[df['label'] == 1]['min_k_score'].mean()
            m0  = df[df['label'] == 0]['min_k_score'].mean()
            dir_ok = m1 > m0
            icon   = '✅' if dir_ok else '⚠️ '
            print(f'  {icon}  direction (m1>m0)  {label}: {m1:.4f} > {m0:.4f}')
            if not dir_ok:
                PASS = False

            expected_sel = df['n_tokens'].apply(lambda n: max(1, math.ceil(n * K_CHECK / 100)))
            sel_ok = (df['n_selected'] == expected_sel).all()
            icon   = '✅' if sel_ok else '❌'
            print(f'  {icon}  n_selected correct {label}')
            if not sel_ok:
                PASS = False

            print()

print()
if PASS:
    print('✅  Module 03 PASSED — proceed to Module 04.')
else:
    print('❌  Module 03 FAILED — fix issues above before running Module 04.')

## [7] Module 04 — Baseline Scoring

Calls `scripts/run_04_baselines.py` to compute all five baseline MIA scores for every **model × length × setting** combination. Scores are written as CSVs to `{DRIVE_04}/` and are consumed directly by Module 05.

**Baseline methods implemented:**

| Method | Type | Description |
|---|---|---|
| PPL | Reference-free | Mean log-probability (Loss Attack, Yeom et al. 2018) |
| Zlib | Reference-free | PPL divided by zlib compression entropy (Carlini et al. 2021) |
| Lowercase | Reference-free | Log-ratio of original vs. lowercased perplexity (Carlini et al. 2021) |
| Neighbor | Reference-free | DetectGPT-style neighbourhood score (Mattern et al. 2023) |
| Smaller Ref | Reference-based | PPL ratio against a smaller same-family model (Carlini et al. 2021) |

**Smaller Ref model pairings** (matching the paper's Table 1):

| Target model | Reference model |
|---|---|
| `pythia-2.8b` | `pythia-70m` |
| `gpt-neo-1.3B` | `gpt-neo-125m` |
| `opt-1.3b` | `opt-350m` |

**Outputs** (written to `{DRIVE_04}/`):

| File | Description |
|---|---|
| `baselines_{model_key}_len{N}_{setting}.csv` | One row per text; columns: `text_id`, `label`, `PPL`, `Zlib`, `Lowercase`, `Neighbor`, `Smaller Ref` |

> **Before running:** confirm Module 03 passed (all ✅ in §6.4).
> Neighbor reruns the target model on perturbed texts and Smaller Ref loads a second model — expect roughly 1.5× the runtime of Module 03.

### 7.1 Smoke Test

Runs the baseline script on 10 rows using the smallest model (`pythia-410m`, length-64 only) to confirm the script loads, scores, and writes a CSV correctly before committing to the full run.

In [ ]:
!PYTHONPATH=/content/ai-final-project python scripts/run_04_baselines.py \
    --input_dir   {DRIVE_01} \
    --logprob_dir {DRIVE_03}/logprobs \
    --output_dir  {DRIVE_04} \
    --models      EleutherAI/pythia-410m \
    --model_keys  pythia410m \
    --lengths     64 \
    --settings    original \
    --smoke

print('\nSmoke test output preview:')
import pandas as pd
_smoke_csv = f'{DRIVE_04}/baselines_pythia410m_len64_original.csv'
df_smoke   = pd.read_csv(_smoke_csv)
display(df_smoke[['text_id', 'label', 'PPL', 'Zlib', 'Lowercase', 'Neighbor', 'Smaller Ref']])

### 7.2 Full Dataset Run

Scores all models across all configured lengths and the `original` setting. Each CSV is written to Drive as it completes. If the run is interrupted, re-run this cell — `--skip_existing` will resume from where it left off without recomputing finished files.

> ⚠️ Add `paraphrase` to `_settings_arg` once that dataset is available.

In [ ]:
_lengths_arg    = ' '.join(str(l) for l in LENGTHS)
_models_arg     = ' '.join(MODELS.values())
_model_keys_arg = ' '.join(MODELS.keys())
_settings_arg   = 'original'   # extend to 'original paraphrase' when ready

!PYTHONPATH=/content/ai-final-project python scripts/run_04_baselines.py \
    --input_dir   {DRIVE_01} \
    --logprob_dir {DRIVE_03}/logprobs \
    --output_dir  {DRIVE_04} \
    --models      {_models_arg} \
    --model_keys  {_model_keys_arg} \
    --lengths     {_lengths_arg} \
    --settings    {_settings_arg} \
    --skip_existing

### 7.3 Output Preview

Shows the per-method member vs. non-member mean score for the first configured length. All five methods should show member mean **higher** than non-member mean — this is the direction check before the formal sanity check below.

In [ ]:
from pathlib import Path
import pandas as pd

_length  = LENGTHS[0]
_setting = 'original'
METHODS  = ['PPL', 'Zlib', 'Lowercase', 'Neighbor', 'Smaller Ref']

for model_key in MODELS:
    csv_path = Path(DRIVE_04) / f'baselines_{model_key}_len{_length}_{_setting}.csv'
    if not csv_path.exists():
        print(f'[MISSING] {csv_path.name}')
        continue

    df = pd.read_csv(csv_path)
    print(f'\n{"="*55}')
    print(f'  {model_key}  |  len={_length}  |  {_setting}  ({len(df)} rows)')
    print(f'{"="*55}')

    for method in METHODS:
        if method not in df.columns:
            print(f'  {method:12s} : [not found in CSV]')
            continue
        m1 = df[df['label'] == 1][method].mean()
        m0 = df[df['label'] == 0][method].mean()
        direction = '✅' if m1 > m0 else '⚠️ '
        print(f'  {method:12s}  member={m1:.4f}  non-member={m0:.4f}  {direction}')

### 7.4 Module 04 Sanity Check

Checks every expected output file across all model × length × setting combinations. Flags missing files, row count mismatches, NaN values, and direction failures. All checks must be green before running Module 05.

**Pass criteria:**
- CSV exists for every model × length × setting
- Row count matches the Module 01 processed CSV for that length
- No NaN values in any score column
- Member mean > non-member mean for all five methods

In [ ]:
from pathlib import Path
import pandas as pd

PASS    = True
METHODS = ['PPL', 'Zlib', 'Lowercase', 'Neighbor', 'Smaller Ref']

for model_key in MODELS:
    for length in LENGTHS:
        for setting in ['original']:   # extend to 'paraphrase' when dataset is ready
            csv_path = Path(DRIVE_04) / f'baselines_{model_key}_len{length}_{setting}.csv'
            ref_csv  = Path(DRIVE_01) / f'wikimia_length{length}_processed.csv'
            label    = f'{model_key} | len={length} | {setting}'

            exists = csv_path.exists()
            print(f'  {"✅" if exists else "❌"}  CSV exists         {label}')
            if not exists:
                PASS = False
                continue

            df = pd.read_csv(csv_path)

            if ref_csv.exists():
                expected = len(pd.read_csv(ref_csv))
                count_ok = len(df) == expected
                print(f'  {"✅" if count_ok else "⚠️ "}  row count          {label}: {len(df)} / {expected}')
                if not count_ok:
                    PASS = False

            for method in METHODS:
                if method not in df.columns:
                    print(f'  ❌  column missing      {label}: {method}')
                    PASS = False
                    continue

                nan_ok = df[method].isna().sum() == 0
                print(f'  {"✅" if nan_ok else "❌"}  no NaN ({method:12s}) {label}')
                if not nan_ok:
                    PASS = False

                m1     = df[df['label'] == 1][method].mean()
                m0     = df[df['label'] == 0][method].mean()
                dir_ok = m1 > m0
                print(f'  {"✅" if dir_ok else "⚠️ "}  direction ({method:12s}) {label}: {m1:.4f} > {m0:.4f}')
                if not dir_ok:
                    PASS = False

            print()

print()
if PASS:
    print('✅  Module 04 PASSED — proceed to Module 05.')
else:
    print('❌  Module 04 FAILED — fix issues above before running Module 05.')

## [8] Module 05 — Evaluation and Figures

Reads the Min-K% scores from Module 03 and the baseline scores from Module 04, computes AUC and TPR@5%FPR for every method, and generates all final deliverables. No models are loaded here — this module is CPU-only and completes in seconds.

**Scripts called and their outputs** (all written to `{DRIVE_05}/`):

| Script | Output | Description |
|---|---|---|
| `run_05_evaluation.py` | `evaluation_summary.csv` | AUC + TPR@5%FPR for every method × model × length × setting |
| `make_table1.py` | `table1_results.csv`, `table1.tex` | Replication of Table 1 from the paper |
| `make_roc_curve.py` | `figures/roc_curve_min_k.png` | ROC curve for Min-K% across all three main models |
| `run_fig2a.py` | `fig2a_results.pkl` | AUC vs. model size sweep (GPU required, ~30 min) |
| `make_fig2a.py` | `figures/fig2a.png` | Replication of Figure 2a from the paper |

> **Before running:** confirm Module 04 passed (all ✅ in §7.4).

### 8.1 Compute Metrics

Merges Min-K% and baseline score CSVs, computes AUC and TPR@5%FPR for every method, and writes `evaluation_summary.csv`. This is the source of truth for all downstream table and figure generation.

In [ ]:
_lengths_arg    = ' '.join(str(l) for l in LENGTHS)
_models_arg     = ' '.join(MODELS.values())
_model_keys_arg = ' '.join(MODELS.keys())
_settings_arg   = 'original'

!python scripts/run_05_evaluation.py \
    --mink_dir      {DRIVE_03} \
    --baseline_dir  {DRIVE_04} \
    --output_dir    {DRIVE_05} \
    --model_keys    {_model_keys_arg} \
    --lengths       {_lengths_arg} \
    --settings      {_settings_arg}

### 8.2 Generate Table 1

Reads `evaluation_summary.csv` and produces a replication of the paper's Table 1 — AUC scores per method per model, with column-wise bolding for the best value. Outputs both a machine-readable CSV and a LaTeX block ready to paste into the report.

In [ ]:
!python scripts/make_table1.py \
    --input_dir     {DRIVE_05} \
    --output_dir    {DRIVE_05} \
    --model_keys    {_model_keys_arg}

print('\nTable 1 preview:')
import pandas as pd
_table_csv = f'{DRIVE_05}/table1_results.csv'
df_table   = pd.read_csv(_table_csv)
display(df_table)

### 8.3 Generate ROC Curve

Plots the ROC curve for Min-K% Prob across all three main models and saves it to `figures/`. The diagonal random baseline is included for reference.

In [ ]:
!python scripts/make_roc_curve.py \
    --input_dir     {DRIVE_05} \
    --output_dir    {DRIVE_05}/figures \
    --model_keys    {_model_keys_arg}

from IPython.display import Image
Image(f'{DRIVE_05}/figures/roc_curve_min_k.png')

### 8.4 Generate Figure 2a — AUC vs. Model Size

Sweeps PPL, Neighbor, and Min-K% across four Pythia sizes (160M → 2.8B) on length-128 texts, replicating Figure 2a from the paper. The `run_fig2a.py` cell reloads all four models and requires GPU — skip this if you only need Table 1 and the ROC curve for now.

> ⚠️ `run_fig2a.py` requires GPU and takes approximately 30 minutes on a T4.

In [ ]:
!python scripts/run_fig2a.py \
    --input_dir     {DRIVE_01} \
    --output_dir    {DRIVE_05} \
    --length        128

!python scripts/make_fig2a.py \
    --input_dir     {DRIVE_05} \
    --output_dir    {DRIVE_05}/figures

from IPython.display import Image
Image(f'{DRIVE_05}/figures/fig2a.png')

### 8.5 Results Summary

Prints the full AUC and TPR@5%FPR table inline for quick inspection. Reads directly from `evaluation_summary.csv` — no recomputation.

In [ ]:
from pathlib import Path
import pandas as pd

summary_path = Path(DRIVE_05) / 'evaluation_summary.csv'

if not summary_path.exists():
    print(f'[MISSING] {summary_path} — run §8.1 first.')
else:
    df_eval = pd.read_csv(summary_path)

    METHODS    = ['Neighbor', 'PPL', 'Zlib', 'Lowercase', 'Smaller Ref', 'Min-K%']
    model_keys = list(MODELS.keys())

    print(f'{"Method":<14}', end='')
    for mk in model_keys:
        print(f'  {mk:>22}', end='')
    print(f'  {"Avg AUC":>8}')
    print('-' * (14 + 24 * len(model_keys) + 10))

    for method in METHODS:
        print(f'{method:<14}', end='')
        aucs = []
        for mk in model_keys:
            row = df_eval[(df_eval['method'] == method) & (df_eval['model_key'] == mk)]
            if row.empty:
                print(f'  {"N/A":>22}', end='')
            else:
                auc = row['auc'].values[0]
                tpr = row['tpr_at_5fpr'].values[0]
                aucs.append(auc)
                print(f'  AUC={auc:.3f} TPR={tpr:.3f}', end='')
        avg = sum(aucs) / len(aucs) if aucs else float('nan')
        print(f'  {avg:>8.3f}')

### 8.6 Module 05 Sanity Check

Verifies all required output files exist, that `evaluation_summary.csv` contains a row for every method × model combination, and checks the paper's central claim: Min-K% AUC should exceed PPL AUC for at least two of the three main models. A ⚠️ on the direction check is not necessarily a bug — it may reflect differences in model size or dataset composition from the paper's setup.

**Pass criteria:**
- `evaluation_summary.csv`, `table1_results.csv`, `table1.tex`, and `roc_curve_min_k.png` all present
- One row per method × model in `evaluation_summary.csv`
- Min-K% AUC > PPL AUC for at least 2 of 3 models

In [ ]:
from pathlib import Path
import pandas as pd

PASS     = True
FIG_DIR  = Path(DRIVE_05) / 'figures'
METHODS  = ['Neighbor', 'PPL', 'Zlib', 'Lowercase', 'Smaller Ref', 'Min-K%']

# ── File existence ─────────────────────────────────────────────────────────
required_files = {
    'evaluation_summary.csv' : Path(DRIVE_05) / 'evaluation_summary.csv',
    'table1_results.csv'     : Path(DRIVE_05) / 'table1_results.csv',
    'table1.tex'             : Path(DRIVE_05) / 'table1.tex',
    'roc_curve_min_k.png'    : FIG_DIR / 'roc_curve_min_k.png',
}
for name, path in required_files.items():
    ok = path.exists()
    print(f'  {"✅" if ok else "❌"}  {name}')
    if not ok:
        PASS = False

# ── evaluation_summary.csv content checks ─────────────────────────────────
summary_path = Path(DRIVE_05) / 'evaluation_summary.csv'
if summary_path.exists():
    df_eval = pd.read_csv(summary_path)

    print()
    for model_key in MODELS:
        for method in METHODS:
            row = df_eval[
                (df_eval['method'] == method) &
                (df_eval['model_key'] == model_key)
            ]
            has_row = len(row) > 0
            print(f'  {"✅" if has_row else "❌"}  row exists: {method:14s} | {model_key}')
            if not has_row:
                PASS = False

    # ── Key claim: Min-K% AUC > PPL AUC ──────────────────────────────────
    print()
    wins = 0
    for model_key in MODELS:
        mink_row = df_eval[(df_eval['method'] == 'Min-K%') & (df_eval['model_key'] == model_key)]
        ppl_row  = df_eval[(df_eval['method'] == 'PPL')    & (df_eval['model_key'] == model_key)]
        if mink_row.empty or ppl_row.empty:
            continue
        mink_auc = mink_row['auc'].values[0]
        ppl_auc  = ppl_row['auc'].values[0]
        ok       = mink_auc > ppl_auc
        wins    += int(ok)
        print(f'  {"✅" if ok else "⚠️ "}  Min-K% > PPL  {model_key}: {mink_auc:.3f} vs {ppl_auc:.3f}')

    claim_ok = wins >= 2
    print(f'\n  {"✅" if claim_ok else "⚠️ "}  Min-K% beats PPL in {wins}/{len(MODELS)} models '
          f'(paper claims consistent improvement)')
    if not claim_ok:
        print('       ⚠️  This may reflect dataset/model differences from the paper, not a bug.')

print()
if PASS:
    print('✅  Module 05 PASSED — all deliverables ready.')
else:
    print('❌  Module 05 FAILED — fix issues above.')